<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/glasslego/ml-deep-learning-study/blob/main/src/deep_learning_basic/xor_detail.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />구글 코랩에서 실행하기</a>
  </td>
</table>

In [10]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import warnings
warnings.filterwarnings("ignore")

# 나눔고딕 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# 폰트 설정
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)  # 마이너스 기호 깨짐 방지

# PyTorch 및 환경 정보 출력
print(f"PyTorch 버전: {torch.__version__}")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-nanum is already the newest version (20200506-1).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.
/usr/share/fonts: caching, new cache contents: 0 fonts, 1 dirs
/usr/share/fonts/truetype: caching, new cache contents: 0 fonts, 3 dirs
/usr/share/fonts/truetype/humor-sans: caching, new cache contents: 1 fonts, 0 dirs
/usr/share/fonts/truetype/liberation: caching, new cache contents: 16 fonts, 0 dirs
/usr/share/fonts/truetype/nanum: caching, new cache contents: 12 fonts, 0 dirs
/usr/local/share/fonts: caching, new cache contents: 0 fonts, 0 dirs
/root/.local/share/fonts: skipping, no such directory
/root/.fonts: skipping, no such directory
/usr/share/fonts/truetype: skipping, looped directory detected
/usr/share/fonts/truetype/humor-sans: skipping, looped directory detected
/usr/share/fonts/truetype/liberation: skipping, looped directory detected
/usr/share/fonts/truetype/

In [19]:
import numpy as np
# ============================================================
# 활성화 함수 정의
# ============================================================
def sigmoid(x):
    """
    시그모이드 함수: 입력값을 0~1 사이로 변환
    - 큰 양수 → 1에 가까워짐
    - 큰 음수 → 0에 가까워짐
    - 0 → 0.5
    """
    return 1 / (1 + np.exp(-x))

In [20]:
# ============================================================
# 2. XOR 데이터 준비
# ============================================================
# 입력 (X): 2개의 입력 (A, B)
X = np.array([[0, 0],
                [0, 1],
                [1, 0],
                [1, 1]])

# 정답 (y): XOR 진리표 (A와 B가 다를 때만 1)
y = np.array([[0],
                [1],
                [1],
                [0]])

print("=" * 70)
print("🧠 XOR 문제를 해결하는 2층 신경망 (수정본)")
print("=" * 70)
print("\n📌 XOR 진리표 (목표):")
print("   A | B | 출력 (y)")
print("  ---|---|----------")
for i in range(4):
    print(f"   {X[i, 0]} | {X[i, 1]} |    {y[i, 0]}")

🧠 XOR 문제를 해결하는 2층 신경망 (수정본)

📌 XOR 진리표 (목표):
   A | B | 출력 (y)
  ---|---|----------
   0 | 0 |    0
   0 | 1 |    1
   1 | 0 |    1
   1 | 1 |    0


In [21]:
# ============================================================
# 3. 네트워크 구조 및 가중치 설정 (핵심)
# ============================================================
print("\n" + "=" * 70)
print("🏗️  네트워크 구조 및 미리 학습된 가중치 설정")
print("=" * 70)
print("""
[입력층 (2개)]  →  [은닉층 (2개)]  →  [출력층 (1개)]
    (A, B)           (h1, h2)           (Output)
""")
print("💡 원리: XOR = (A OR B) AND (NOT (A AND B))")
print("   - 은닉층의 h1이 'AND' 게이트 역할을 합니다.")
print("   - 은닉층의 h2가 'OR' 게이트 역할을 합니다.")
print("   - 출력층이 h1과 h2를 조합해 'h2 AND (NOT h1)'을 계산합니다.")
print("-" * 70)


🏗️  네트워크 구조 및 미리 학습된 가중치 설정

[입력층 (2개)]  →  [은닉층 (2개)]  →  [출력층 (1개)]
    (A, B)           (h1, h2)           (Output)

💡 원리: XOR = (A OR B) AND (NOT (A AND B))
   - 은닉층의 h1이 'AND' 게이트 역할을 합니다.
   - 은닉층의 h2가 'OR' 게이트 역할을 합니다.
   - 출력층이 h1과 h2를 조합해 'h2 AND (NOT h1)'을 계산합니다.
----------------------------------------------------------------------


In [22]:
# ------------------------------------------------------------
# 가중치 1: 입력층 -> 은닉층 (W1, b1)
# ------------------------------------------------------------
# W1[i, j]: i번째 입력이 j번째 은닉 뉴런으로 가는 가중치
#       (h1으로) (h2로)
W1 = np.array([[20, 20],  # A -> h1 (20), A -> h2 (20)
                [20, 20]]) # B -> h1 (20), B -> h2 (20)

# b1[j]: j번째 은닉 뉴런의 편향
#       (h1용) (h2용)
b1 = np.array([-30, -10])

print("⚙️ [은닉층] 가중치 및 편향:")
print("   h1 (AND 게이트): z = 20*A + 20*B - 30")
print("     (0,0) → z=-30 → σ(z) ≈ 0 🔴")
print("     (0,1) → z=-10 → σ(z) ≈ 0 🔴")
print("     (1,0) → z=-10 → σ(z) ≈ 0 🔴")
print("     (1,1) → z=+10 → σ(z) ≈ 1 🟢")
print("\n   h2 (OR 게이트): z = 20*A + 20*B - 10")
print("     (0,0) → z=-10 → σ(z) ≈ 0 🔴")
print("     (0,1) → z=+10 → σ(z) ≈ 1 🟢")
print("     (1,0) → z=+10 → σ(z) ≈ 1 🟢")
print("     (1,1) → z=+30 → σ(z) ≈ 1 🟢")

print("-" * 70)

⚙️ [은닉층] 가중치 및 편향:
   h1 (AND 게이트): z = 20*A + 20*B - 30
     (0,0) → z=-30 → σ(z) ≈ 0 🔴
     (0,1) → z=-10 → σ(z) ≈ 0 🔴
     (1,0) → z=-10 → σ(z) ≈ 0 🔴
     (1,1) → z=+10 → σ(z) ≈ 1 🟢

   h2 (OR 게이트): z = 20*A + 20*B - 10
     (0,0) → z=-10 → σ(z) ≈ 0 🔴
     (0,1) → z=+10 → σ(z) ≈ 1 🟢
     (1,0) → z=+10 → σ(z) ≈ 1 🟢
     (1,1) → z=+30 → σ(z) ≈ 1 🟢
----------------------------------------------------------------------


In [23]:
# ------------------------------------------------------------
# 가중치 2: 은닉층 -> 출력층 (W2, b2)
# ------------------------------------------------------------
# W2[i]: i번째 은닉 뉴런(h)이 출력 뉴런으로 가는 가중치
W2 = np.array([[-20],  # h1 -> output (-20)
                [20]])   # h2 -> output (20)

# b2: 출력 뉴런의 편향
b2 = np.array([-10])

print("⚙️ [출력층] 가중치 및 편향 (XOR 구현):")
print("   z = h1*(-20) + h2*(20) - 10   (즉, h2 AND (NOT h1))")
print("   [h1, h2] (A,B) |  계산 (z)         | 출력 (σ(z)) | XOR 정답")
print("   [~0, ~0] (0,0) | 0*(-20)+0*(20)-10 = -10 |     ~0      |    0")
print("   [~0, ~1] (0,1) | 0*(-20)+1*(20)-10 = +10 |     ~1      |    1")
print("   [~0, ~1] (1,0) | 0*(-20)+1*(20)-10 = +10 |     ~1      |    1")
print("   [~1, ~1] (1,1) | 1*(-20)+1*(20)-10 = -10 |     ~0      |    0")

⚙️ [출력층] 가중치 및 편향 (XOR 구현):
   z = h1*(-20) + h2*(20) - 10   (즉, h2 AND (NOT h1))
   [h1, h2] (A,B) |  계산 (z)         | 출력 (σ(z)) | XOR 정답
   [~0, ~0] (0,0) | 0*(-20)+0*(20)-10 = -10 |     ~0      |    0
   [~0, ~1] (0,1) | 0*(-20)+1*(20)-10 = +10 |     ~1      |    1
   [~0, ~1] (1,0) | 0*(-20)+1*(20)-10 = +10 |     ~1      |    1
   [~1, ~1] (1,1) | 1*(-20)+1*(20)-10 = -10 |     ~0      |    0


In [26]:
# ============================================================
# 4. 순전파 (Forward Propagation) - 전체 데이터 동시 계산
# ============================================================
print("\n" + "=" * 70)
print("🚀 순전파 (Forward Propagation) - 4개 케이스 동시 계산")
print("=" * 70)

# [Step 1] 입력층 -> 은닉층 계산
# X (4,2) @ W1 (2,2) = (4,2)
# z1 = (입력 * 가중치) + 편향
z1 = np.dot(X, W1) + b1
# h = 은닉층의 최종 출력 (활성화 함수 적용)
h = sigmoid(z1)

print("[Step 1] 은닉층 출력 (h):")
print("   입력 (A,B) |  z1 [h1, h2] | h [h1(AND), h2(OR)]")
print("  -----------|--------------|---------------------")
for i in range(4):
    print(f"   ({X[i, 0]}, {X[i, 1]})    | {z1[i]} | [{h[i, 0]:.3f}, {h[i, 1]:.3f}]")

# [Step 2] 은닉층 -> 출력층 계산
# h (4,2) @ W2 (2,1) = (4,1)
# z2 = (은닉층 출력 * 가중치) + 편향
z2 = np.dot(h, W2) + b2
# output = 신경망의 최종 출력 (활성화 함수 적용)
output = sigmoid(z2)

print("\n[Step 2] 최종 출력층 계산 (output):")
print("   h [h1, h2] |     z2 (입력) | output (σ(z2))")
print("  -------------|---------------|----------------")
for i in range(4):
    print(f"   [{h[i, 0]:.3f}, {h[i, 1]:.3f}] | {z2[i, 0]:9.3f} |    {output[i, 0]:.5f}")

# [Step 3] 예측 (Prediction)
# 0.5를 기준으로 0 또는 1로 변환
predictions = (output > 0.5).astype(int)


🚀 순전파 (Forward Propagation) - 4개 케이스 동시 계산
[Step 1] 은닉층 출력 (h):
   입력 (A,B) |  z1 [h1, h2] | h [h1(AND), h2(OR)]
  -----------|--------------|---------------------
   (0, 0)    | [-30 -10] | [0.000, 0.000]
   (0, 1)    | [-10  10] | [0.000, 1.000]
   (1, 0)    | [-10  10] | [0.000, 1.000]
   (1, 1)    | [10 30] | [1.000, 1.000]

[Step 2] 최종 출력층 계산 (output):
   h [h1, h2] |     z2 (입력) | output (σ(z2))
  -------------|---------------|----------------
   [0.000, 0.000] |    -9.999 |    0.00005
   [0.000, 1.000] |     9.998 |    0.99995
   [0.000, 1.000] |     9.998 |    0.99995
   [1.000, 1.000] |    -9.999 |    0.00005


In [27]:
# ============================================================
# 5. 최종 결과 요약
# ============================================================
print("\n" + "=" * 70)
print("📊 최종 결과 요약")
print("=" * 70)
print("   A | B | 정답 (y) | h1 (AND) | h2 (OR) | 최종출력 (≈) | 예측 (Pred) | 결과")
print("  ---|---|----------|----------|---------|--------------|-------------|------")

for i in range(4):
    is_correct = predictions[i] == y[i]
    emoji = "✅" if is_correct else "❌"
    print(f"   {X[i, 0]} | {X[i, 1]} |    {y[i, 0]}     |   {h[i, 0]:.2f}   |  {h[i, 1]:.2f}  |    {output[i, 0]:.3f}     |      {predictions[i, 0]}      |  {emoji}")

print("\n" + "=" * 70)
print("🎉 XOR 문제 해결 완료!")
print("비선형 문제(XOR)를 해결하기 위해 은닉층(h1, h2)이")
print("입력 데이터를 '선형 분리 가능한' 공간으로 변환했습니다.")
print("=" * 70)


📊 최종 결과 요약
   A | B | 정답 (y) | h1 (AND) | h2 (OR) | 최종출력 (≈) | 예측 (Pred) | 결과
  ---|---|----------|----------|---------|--------------|-------------|------
   0 | 0 |    0     |   0.00   |  0.00  |    0.000     |      0      |  ✅
   0 | 1 |    1     |   0.00   |  1.00  |    1.000     |      1      |  ✅
   1 | 0 |    1     |   0.00   |  1.00  |    1.000     |      1      |  ✅
   1 | 1 |    0     |   1.00   |  1.00  |    0.000     |      0      |  ✅

🎉 XOR 문제 해결 완료!
비선형 문제(XOR)를 해결하기 위해 은닉층(h1, h2)이
입력 데이터를 '선형 분리 가능한' 공간으로 변환했습니다.
